# 🧬 Master Pipeline Project 5
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup
Collega il Google Drive ed esegui il setup iniziale. Ogni componente salvato in `/content/drive/MyDrive/AML/` sarà persistente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b osagie5 https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data

## 🔍 1. Stage 1: Backbone Analysis
Analisi zero-shot di DINOv2 a diverse profondità.

In [ ]:
# 1.1 Baseline Standard (Ultimo Layer)
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --results_file /content/drive/MyDrive/AML/Results/baseline.txt

In [ ]:
# 1.2 Analisi Layer-wise (Layer 4, 8, 11)
for layer in [4, 8, 11]:
    print(f"\n--- Valutazione Layer {layer} ---")
    !python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --layer {layer} --results_file /content/drive/MyDrive/AML/Results/layer_{layer}.txt

## 🚀 2. Stage 2: Fine-Tuning Efficiente (Ablation Study)
Confrontiamo l'addestramento con e senza Curriculum Learning.

In [ ]:
# 2.1 Solo LoRA (Senza Curriculum)
!python train.py --epochs 5 --curriculum_epochs 0 --num_workers 4 --output_dir ./checkpoints/lora_only --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_only

In [ ]:
# 2.2 LoRA + Curriculum (Final Model)
!python train.py --epochs 5 --curriculum_epochs 2 --num_workers 4 --output_dir ./checkpoints/lora_curriculum --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_curriculum

## 🎯 3. Stage 3: Raffinamento Adattivo
Ablation study dell'Adaptive Windowing.

In [ ]:
CKPT = "/content/drive/MyDrive/AML/Checkpoints/lora_curriculum/best.pth"
print("--- PCK CON Adaptive Window ---")
!python evaluate.py --checkpoint {CKPT} --results_file /content/drive/MyDrive/AML/Results/stage3_with.txt
print("\n--- PCK SENZA Adaptive Window ---")
!python evaluate.py --checkpoint {CKPT} --no_adaptive_win --results_file /content/drive/MyDrive/AML/Results/stage3_without.txt

## 🧑‍💻 4. Stage 4: Super Demo Interattiva
Esegui questa cella per lanciare l'interfaccia UI direttamente in Colab.

In [ ]:
# ── Stage 4: Demo Interattiva (Gradio) ─────────────────────────
import torch
import os
import gradio as gr
from PIL import Image, ImageDraw
import torchvision.transforms as T
from models.extractor import DINOv2Extractor
from models.lora import apply_lora_to_dinov2
from models.correspondence import SemanticCorrespondenceModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

MODELS_CACHE = {}

def get_model(ckpt_path=None):
    if ckpt_path in MODELS_CACHE: return MODELS_CACHE[ckpt_path]
    backbone = DINOv2Extractor(model_name="dinov2_vitb14", freeze=True)
    if ckpt_path:
        ckpt = torch.load(ckpt_path, map_location=device)
        backbone.model = apply_lora_to_dinov2(backbone.model, rank=ckpt["args"].get("lora_rank", 16))
        model = SemanticCorrespondenceModel(backbone=backbone, use_adaptive_win=True).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model = SemanticCorrespondenceModel(backbone=backbone, use_adaptive_win=True).to(device)
    model.eval()
    MODELS_CACHE[ckpt_path] = model
    return model

def predict(src_img, trg_img, x, y, ckpt_name, use_aw):
    checkpoint_base = "/content/drive/MyDrive/AML/Checkpoints"
    path = os.path.join(checkpoint_base, ckpt_name, "best.pth") if "Baseline" not in ckpt_name else None
    if path and not os.path.exists(path): path = None
    
    m = get_model(path)
    m.use_adaptive_win = use_aw
    s_t = transform(src_img).unsqueeze(0).to(device)
    t_t = transform(trg_img).unsqueeze(0).to(device)
    scale = (224 / src_img.width, 224 / src_img.height)
    src_kp = torch.tensor([[[x * scale[0], y * scale[1]]]], device=device).float()
    out = m(s_t, t_t, src_kps=src_kp)
    pkp = out["pred_kps"][0,0].cpu().numpy()
    return pkp[0] * (trg_img.width/224), pkp[1] * (trg_img.height/224)

def gradio_fn(src_img, trg_img, ckpt, aw, evt: gr.SelectData):
    tx, ty = predict(src_img, trg_img, evt.index[0], evt.index[1], ckpt, aw)
    s_o, t_o = src_img.copy(), trg_img.copy()
    ImageDraw.Draw(s_o).ellipse([evt.index[0]-6, evt.index[1]-6, evt.index[0]+6, evt.index[1]+6], fill="red", outline="white")
    ImageDraw.Draw(t_o).ellipse([tx-6, ty-6, tx+6, ty+6], fill="green", outline="white")
    return s_o, t_o

# Menu Checkpoints
ckpt_options = ["Baseline (No LoRA)"]
if os.path.exists("/content/drive/MyDrive/AML/Checkpoints"):
    dirs = [d for d in os.listdir("/content/drive/MyDrive/AML/Checkpoints") if os.path.isdir(os.path.join("/content/drive/MyDrive/AML/Checkpoints", d))]
    ckpt_options += sorted(dirs)

with gr.Blocks() as demo:
    gr.Markdown("## 🐈 Demo Interattiva: Multi-Stage Correspondence")
    with gr.Row():
        ckpt = gr.Dropdown(choices=ckpt_options, value=ckpt_options[0], label="Modello")
        aw = gr.Checkbox(value=True, label="Adaptive Window (Stage 3)")
    with gr.Row():
        s_in = gr.Image(type="pil", label="Source (Clicca un punto)")
        t_in = gr.Image(type="pil", label="Target")
    with gr.Row():
        s_out = gr.Image(type="pil", label="Source Result")
        t_out = gr.Image(type="pil", label="Target Result")
    s_in.select(gradio_fn, [s_in, t_in, ckpt, aw], [s_out, t_out])

demo.launch(share=True, inline=False)